In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("cleaned_shots_logs.csv")

In [3]:
df.head()

,GAME_ID,MATCHUP,LOCATION,W,FINAL_MARGIN,SHOT_NUMBER,PERIOD,GAME_CLOCK,SHOT_CLOCK,DRIBBLES,...,SHOT_DIST,PTS_TYPE,SHOT_RESULT,CLOSEST_DEFENDER,CLOSEST_DEFENDER_PLAYER_ID,CLOSE_DEF_DIST,FGM,PTS,player_name,player_id
0,21400899,"MAR 04, 2015 - CHA @ BKN",A,W,24,1,1,69,10.8,2,...,7.7,2,made,"ANDERSON, ALAN",101187,1.3,1,2,"ROBERTS, BRIAN",203148
1,21400899,"MAR 04, 2015 - CHA @ BKN",A,W,24,2,1,14,3.4,0,...,28.2,3,missed,"BOGDANOVIC, BOJAN",202711,6.1,0,0,"ROBERTS, BRIAN",203148
2,21400899,"MAR 04, 2015 - CHA @ BKN",A,W,24,3,1,0,0.0,3,...,10.1,2,missed,"BOGDANOVIC, BOJAN",202711,0.9,0,0,"ROBERTS, BRIAN",203148
3,21400899,"MAR 04, 2015 - CHA @ BKN",A,W,24,4,2,707,10.3,2,...,17.2,2,missed,"BROWN, MARKEL",203900,3.4,0,0,"ROBERTS, BRIAN",203148
4,21400899,"MAR 04, 2015 - CHA @ BKN",A,W,24,5,2,634,10.9,2,...,3.7,2,missed,"YOUNG, THADDEUS",201152,1.1,0,0,"ROBERTS, BRIAN",203148


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122766 entries, 0 to 122765
Data columns (total 21 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   GAME_ID                     122766 non-null  int64  
 1   MATCHUP                     122766 non-null  object 
 2   LOCATION                    122766 non-null  object 
 3   W                           122766 non-null  object 
 4   FINAL_MARGIN                122766 non-null  int64  
 5   SHOT_NUMBER                 122766 non-null  int64  
 6   PERIOD                      122766 non-null  int64  
 7   GAME_CLOCK                  122766 non-null  int64  
 8   SHOT_CLOCK                  122766 non-null  float64
 9   DRIBBLES                    122766 non-null  int64  
 10  TOUCH_TIME                  122766 non-null  float64
 11  SHOT_DIST                   122766 non-null  float64
 12  PTS_TYPE                    122766 non-null  int64  
 13  SHOT_RESULT   

Before modelling the data we need to understand what features should be used and what result we are expecting. We are trying to predict which shots will be made under certain conditions. For that we have two features`FGM` and `SHOT_RESULT`. `FGM` is already in categorical form of 0/1; however `SHOT_RESULT` isn't. So we will take `FGM` as the result. We will only be working with numeric features, excluding `player_id` and `game_id` since they have no use in prediction. We will also not be taking any categorical data since they don't contribute to the final result. Now that features have been selected we will need to see the distribution of the result.

In [5]:
df["FGM"].value_counts(normalize=True)

FGM
0    0.547937
1    0.452063
Name: proportion, dtype: float64

In [6]:
x_column = ["CLOSE_DEF_DIST", "SHOT_NUMBER", "PERIOD", "GAME_CLOCK", "SHOT_CLOCK", "DRIBBLES", "TOUCH_TIME","SHOT_DIST"]

df_model = df[x_column]

The result is distributed in a balanced format, so we will not be conducting any under-- or oversampling of the data. 

### Logistic Regression

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
#base logistic model
X = df_model
y = df["FGM"]
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.3, random_state=18)
#train data
log_model = LogisticRegression()
log_model.fit(X_train, y_train)
log_model.predict(X_train)
#test data
y_base_pred = log_model.predict(X_test)
score = log_model.score(X_test,y_test)
print(score)
print(classification_report(y_test,y_base_pred))


0.6093945153407548
              precision    recall  f1-score   support

           0       0.62      0.73      0.67     20163
           1       0.59      0.46      0.52     16667

    accuracy                           0.61     36830
   macro avg       0.60      0.60      0.59     36830
weighted avg       0.61      0.61      0.60     36830



C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


We have pretty decent results. The total accuracy of the model is 61%. The model predicted 62% of missed shots correctly and 59% of made shots. It also predicted 73% of missed shots accurately, while it only predicted 46% of made shots accurately. Now let's try to fine-tune the model.

In [8]:
from sklearn.preprocessing import StandardScaler

X = df_model
y = df["FGM"]
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.3, random_state=18)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
#train data
log_model_scaled = LogisticRegression(class_weight="balanced", max_iter = 1000)
log_model_scaled.fit(X_train_scaled, y_train)
log_model_scaled.predict(X_train_scaled)
#test data
y_scaled_pred = log_model_scaled.predict(X_test_scaled)
train_score = log_model_scaled.score(X_train_scaled, y_train)
test_score = log_model_scaled.score(X_test_scaled,y_test)
print(train_score, test_score)
print(classification_report(y_test,y_scaled_pred))

0.5893455594861292 0.5908498506652186
              precision    recall  f1-score   support

           0       0.63      0.60      0.62     20163
           1       0.55      0.58      0.56     16667

    accuracy                           0.59     36830
   macro avg       0.59      0.59      0.59     36830
weighted avg       0.59      0.59      0.59     36830



In [9]:
import pickle
# Save model
with open("./api/log_model.pkl", "wb") as f:
    pickle.dump(log_model_scaled, f)

# Save scaler
with open("./api/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and scaler saved!")

Model and scaler saved!
